# Loss_Study — Validazione COMPLETA (un solo run, ~5-7 min)

Carica il checkpoint **una volta** ed esegue tutto in sequenza:
1. **Sicurezza closed-loop** (scenari avversari) → 4 grafici + 4 CSV
2. **Vetrina**: accuracy, raster spike, energia SNN vs ANN, animazione auto (GIF), dashboard

Il **dashboard** finale include anche il verdetto di sicurezza (prodotto al passo 1).
Niente training → tempi brevi. Checkpoint solo su Azure; le celle col modello saltano se assente.


In [ ]:
# Cell 1 -- ENV + cartelle
import sys, os, subprocess
import importlib.util as _imu
ANALYSIS = 'v1_realistic_cutin'   # nome analisi -> results/evaluate/<ANALYSIS>/{Eval_ClosedLoop,Showcase}
EVAL_DIR = f'results/evaluate/{ANALYSIS}/Eval_ClosedLoop'
SHOW_DIR = f'results/evaluate/{ANALYSIS}/Showcase'
BRANCH = 'Loss_Study'
_TMP_MSG = '/tmp/valfull.txt' if os.path.isdir('/tmp') else 'valfull.txt'
for d in (EVAL_DIR, SHOW_DIR):
    os.makedirs(d, exist_ok=True)
for pkg in ['pandas', 'matplotlib', 'pillow']:
    if _imu.find_spec(pkg.replace('pillow', 'PIL')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
br = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip()
assert br == BRANCH, f'branch={br} != {BRANCH}'
print('[ValidationFull] ENV OK — eval + showcase in un solo run')

In [ ]:
# Cell 2 -- carica checkpoint -> models (eval) + model primario (showcase)
import os, torch, shutil
from core.network import build_model
CANDIDATES = [
    ('S3 d0.3 (launch)', 'checkpoints/LS3_PEAK_R0_launch_d03/best_model.pt', 32, 8),
    ('S3 d1.0 (launch)', 'checkpoints/LS3_PEAK_R0_launch/best_model.pt',     32, 8),
    ('R33_C2 CLEAN',     'checkpoints/R33_C2_A1_T12_fix/best_model.pt',       32, 8),
]
S3_LOG = 'results/Loss_Study/S3/PEAK/LS3_PEAK_R0_launch_d03/training_log.csv'
CKPT = CANDIDATES[0][1]
models = {}
for label, path, h, rk in CANDIDATES:
    if not os.path.isfile(path):
        print(f'  [skip] {label}: checkpoint assente ({path})'); continue
    ck = torch.load(path, map_location='cpu', weights_only=False)
    m = build_model(variant='baseline', hidden_size=h, rank=rk, max_delay=6, bit_shift=3)
    m.load_state_dict(ck['model_state']); m.eval(); models[label] = m
    dst = f'{EVAL_DIR}/checkpoints/{label.replace(" ", "_").replace("(", "").replace(")", "")}.pt'
    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(path, dst)
    print(f'  [OK] {label} ({sum(p.numel() for p in m.parameters())} param)')
model = next(iter(models.values())) if models else None     # showcase usa il primario
print(f'Modelli eval: {list(models.keys())} (+ oracolo) | showcase: {list(models.keys())[0] if models else "NESSUNO"}')
if not models:
    print('NB: nessun checkpoint -> eval gira solo oracolo; showcase fa accuracy+dashboard.')

In [ ]:
# Cell 3 -- Esegui validazione closed-loop: modelli (+oracolo) x driver x scenari
import numpy as np, pandas as pd
from data.generator import _sample_scenario, parse_scenario_mix
from utils.closed_loop_eval import simulate, build_scenarios, all_metrics, string_stability_gain

N_DRIVERS = 20          # scenari-guidatore campionati (params veri diversi)
N_STEPS = 600
MIX = parse_scenario_mix('highway:0.4,urban:0.3,truck:0.2,mixed:0.1')
rng = np.random.default_rng(42)

# campiona N_DRIVERS set di parametri veri (diversi tipi di guidatore)
drivers = []
_r = np.random.default_rng(7)
for _ in range(N_DRIVERS):
    p, prof, stype, ci = _sample_scenario(_r, scenario_mix=MIX)
    drivers.append((stype, np.array([p['v0'], p['T'], p['s0'], p['a'], p['b']], dtype=np.float32)))

sources = [('oracle', None)] + [(lbl, m) for lbl, m in models.items()]
rows = []
for di, (dtype, pgt) in enumerate(drivers):
    scen = build_scenarios(pgt, N=N_STEPS, rng=np.random.default_rng(100 + di))
    for sname, vl, s_i, v_i, cut in scen:
        for src, mdl in sources:
            tr = simulate(mdl, pgt, vl, s_i, v_i, cut_in=cut)
            mt = all_metrics(tr)
            mt['gain'] = string_stability_gain(tr) if sname == 'sinusoidal' else float('nan')
            mt.update({'source': src, 'scenario': sname, 'driver': di, 'driver_type': dtype})
            rows.append(mt)
df = pd.DataFrame(rows)
df.to_csv(f'{EVAL_DIR}/closedloop_metrics_raw.csv', index=False)
print(f'Simulazioni: {len(df)} ({len(sources)} sorgenti x {N_DRIVERS} driver x 5 scenari)')

In [ ]:
# Cell 4 -- Sintesi sicurezza + confronto SNN vs oracolo
import pandas as pd, numpy as np
from IPython.display import display, Markdown

display(Markdown('## Verdetto SICUREZZA per sorgente (su tutti gli scenari)'))
safety = df.groupby('source').agg(
    n=('collided', 'size'),
    collision_rate=('collided', 'mean'),
    worst_min_gap=('min_gap', 'min'),
    worst_min_ttc=('min_ttc', lambda x: np.min(x[np.isfinite(x)]) if np.isfinite(x).any() else np.inf),
    worst_min_headway=('min_time_headway', lambda x: np.min(x[np.isfinite(x)]) if np.isfinite(x).any() else np.inf),
    max_DRAC=('max_DRAC', 'max'),
    mean_TET=('TET', 'mean'), mean_TIT=('TIT', 'mean'),
).round(3)
# Wilson 95% CI superiore sul collision_rate (proporzione) — onesta sul campione N
def _wilson_hi(p, n, z=1.96):
    if n == 0: return float('nan')
    c = z * z / n
    return round(((p + c/2) + z*np.sqrt(p*(1-p)/n + c/(4*n))) / (1 + c), 3)
safety['collision_CI95hi'] = [_wilson_hi(p, n) for p, n in zip(safety['collision_rate'], safety['n'])]
safety.to_csv(f'{EVAL_DIR}/safety_summary.csv')
display(safety)
print('CRITERIO: collision_rate ~ 0 (con CI). worst_min_ttc > 1.5s ideale. DRAC < ~9 (>9=inevitabile).')
print(f'N={int(safety["n"].iloc[0])} sim/sorgente. collision_CI95hi = limite superiore 95% (Wilson) del collision rate.')

display(Markdown('## Collision rate per scenario (dove la rete cede?)'))
coll = df.pivot_table(index='scenario', columns='source', values='collided', aggfunc='mean').round(3)
coll.to_csv(f'{EVAL_DIR}/collision_by_scenario.csv')
display(coll)

display(Markdown('## Comfort + tracking + string stability (medie)'))
qual = df.groupby('source').agg(
    rms_accel=('rms_accel', 'mean'), max_decel=('max_decel', 'max'),
    rms_jerk=('rms_jerk', 'mean'), rms_gap_error=('rms_gap_error', 'mean'),
    string_gain=('gain', 'mean'),
).round(3)
qual.to_csv(f'{EVAL_DIR}/quality_summary.csv')
display(qual)
print('string_gain < 1 = string-stable (smorza le perturbazioni del leader).')
print('Sintesi salvate: safety_summary.csv, collision_by_scenario.csv, quality_summary.csv')

In [ ]:
# Cell 5 -- Set completo: G1 traiettorie+accel, G2 TTC, G3 distribuzione margini, G4 string-stability
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from utils.closed_loop_eval import simulate, build_scenarios, TTC_STAR

sources = [('oracle', None)] + [(lbl, m) for lbl, m in models.items()]
dtype0, pgt0 = next((d for d in drivers if d[0] == 'highway'), drivers[0])
scen0 = {s[0]: s for s in build_scenarios(pgt0, N=N_STEPS, rng=np.random.default_rng(100))}

# G1: gap / velocita / accelerazione per gli scenari dinamici (come reagisce l'ego)
dyn = ['cut_in', 'hard_brake', 'stop_and_go']
fig, axes = plt.subplots(3, len(dyn), figsize=(5 * len(dyn), 11), sharex='col')
for col, sname in enumerate(dyn):
    _, vl, s_i, v_i, cut = scen0[sname]
    for src, mdl in sources:
        tr = simulate(mdl, pgt0, vl, s_i, v_i, cut_in=cut)
        t = np.arange(len(tr['s'])) * 0.1
        axes[0, col].plot(t, tr['s'], label=src, alpha=0.85)
        axes[1, col].plot(t, tr['v'], label=src, alpha=0.85)
        axes[2, col].plot(t, tr['a_ego'], label=src, alpha=0.85)
    axes[1, col].plot(np.arange(len(vl)) * 0.1, vl, 'k--', lw=1, alpha=0.5, label='leader')
    axes[0, col].axhline(0, color='r', lw=1.2); axes[0, col].set_title(sname)
    axes[2, col].axhline(-9, color='r', ls=':', lw=0.8)
    axes[0, col].set_ylabel('gap [m]'); axes[1, col].set_ylabel('v [m/s]'); axes[2, col].set_ylabel('a_ego [m/s2]')
    axes[2, col].set_xlabel('t [s]')
    for r in range(3):
        axes[r, col].grid(alpha=0.3); axes[r, col].legend(fontsize=6)
fig.suptitle('G1 — Traiettorie closed-loop: gap / velocita / accelerazione (rosso = collisione / -9 limite)')
fig.tight_layout(); fig.savefig(f'{EVAL_DIR}/eval_G1_trajectories.png', dpi=120); plt.show()

# G2: TTC(t) negli scenari critici — il segnale di pericolo reale
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for col, sname in enumerate(['cut_in', 'hard_brake']):
    _, vl, s_i, v_i, cut = scen0[sname]
    for src, mdl in sources:
        tr = simulate(mdl, pgt0, vl, s_i, v_i, cut_in=cut)
        ttc = np.where(tr['dv'] > 1e-3, tr['s'] / np.maximum(tr['dv'], 1e-6), np.nan)
        axes[col].plot(np.arange(len(ttc)) * 0.1, np.clip(ttc, 0, 8), label=src, alpha=0.85)
    axes[col].axhline(TTC_STAR, color='r', ls='--', label=f'TTC critico {TTC_STAR}s')
    axes[col].set_title(f'{sname}: TTC(t)'); axes[col].set_xlabel('t [s]'); axes[col].set_ylabel('TTC [s] (clip 8)')
    axes[col].legend(fontsize=7); axes[col].grid(alpha=0.3)
fig.suptitle('G2 — Time-To-Collision: sotto la linea rossa = pericolo')
fig.tight_layout(); fig.savefig(f'{EVAL_DIR}/eval_G2_ttc.png', dpi=120); plt.show()

# G3: distribuzione margini su TUTTE le sim (la CODA = caso peggiore, conta per la safety)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
for src in df['source'].unique():
    sub = df[df['source'] == src]
    a1.scatter(sub['min_gap'], np.clip(sub['min_ttc'], 0, 10), label=src, alpha=0.6, s=28)
a1.axvline(0, color='r', lw=1); a1.axhline(TTC_STAR, color='r', ls='--')
a1.set_xlabel('min_gap [m] (0 = collisione)'); a1.set_ylabel('min_TTC [s] (clip 10)')
a1.set_title('Ogni punto = una sim. Basso-sx = pericoloso (la coda)'); a1.legend(fontsize=8); a1.grid(alpha=0.3)
df.pivot_table(index='scenario', columns='source', values='min_gap', aggfunc='min').plot(kind='bar', ax=a2)
a2.axhline(0, color='r', lw=1.2); a2.set_ylabel('worst min_gap [m]')
a2.set_title('Margine minimo per scenario'); a2.set_xticklabels(a2.get_xticklabels(), rotation=0)
a2.grid(alpha=0.3, axis='y')
fig.suptitle('G3 — Margini di sicurezza: distribuzione (coda) + worst-case per scenario')
fig.tight_layout(); fig.savefig(f'{EVAL_DIR}/eval_G3_margins.png', dpi=120); plt.show()

# G4: string stability — indice di SMORZAMENTO D = std(v_ego)/std(v_leader) (<1 = smorza).
# Annotato su sinusoidale E stop&go (richiesta: indice di smorzamento sullo stop&go).
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for col, sname in enumerate(['sinusoidal', 'stop_and_go']):
    _, vl, s_i, v_i, _ = scen0[sname]
    t = np.arange(len(vl)) * 0.1
    axes[col].plot(t, vl, 'k--', lw=1.6, label='leader')
    for src, mdl in sources:
        tr = simulate(mdl, pgt0, vl, s_i, v_i)
        ve = tr['v']
        D = float(np.std(ve - ve.mean()) / (np.std(vl - vl.mean()) + 1e-9))   # indice di smorzamento
        axes[col].plot(np.arange(len(ve)) * 0.1, ve, alpha=0.85, label=f'{src}  D={D:.2f}')
    axes[col].set_xlabel('t [s]'); axes[col].set_ylabel('v [m/s]')
    axes[col].set_title(f'{sname}: D = std_ego/std_leader  (<1 = smorza, stabile)')
    axes[col].legend(fontsize=7); axes[col].grid(alpha=0.3)
fig.suptitle('G4 — String stability + indice di smorzamento D (sinusoidale e stop&go)')
fig.tight_layout(); fig.savefig(f'{EVAL_DIR}/eval_G4_string_stability.png', dpi=120); plt.show()
print('Salvati 4 grafici: G1 traiettorie+accel, G2 TTC, G3 margini, G4 string-stability+smorzamento (D)')

In [ ]:
# Cell 3 -- ACCURACY scorecard (visiva, dai results S3 d0.3)
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
CH = ['v0', 'T', 's0', 'a', 'b']; TRUE = {'v0': 33.3, 'T': 1.2, 's0': 2.5, 'a': 1.1, 'b': 1.5}
if os.path.isfile(S3_LOG):
    e = pd.read_csv(S3_LOG); i = int(e['val_data'].idxmin())
    nrmse = [float(e[f'val_{c}_nrmse'].iloc[i]) for c in CH]
    pred = [float(e[f'val_{c}_pred_mean'].iloc[i]) for c in CH]
    acc = [max(0.0, 1 - n) * 100 for n in nrmse]    # "accuracy" % ~ 1-NRMSE
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    bars = a1.bar(CH, acc, color=['tab:green' if x > 75 else 'tab:orange' if x > 65 else 'tab:red' for x in acc])
    a1.set_ylim(0, 100); a1.set_ylabel('accuracy ~ (1-NRMSE) [%]'); a1.axhline(75, color='gray', ls=':')
    a1.set_title(f'Accuracy identificazione per parametro (media {np.mean(acc):.0f}%)')
    for b, n in zip(bars, nrmse): a1.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'NRMSE\n{n:.2f}', ha='center', fontsize=8)
    a1.grid(alpha=0.3, axis='y')
    x = np.arange(len(CH)); w = 0.35
    a2.bar(x-w/2, pred, w, label='predetto', color='tab:blue')
    a2.bar(x+w/2, [TRUE[c] for c in CH], w, label='vero', color='tab:gray')
    a2.set_xticks(x); a2.set_xticklabels(CH); a2.set_yscale('log'); a2.set_ylabel('valore (log)')
    a2.set_title('Parametro predetto vs vero'); a2.legend(); a2.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.savefig(f'{SHOW_DIR}/showcase_accuracy.png', dpi=120); plt.show()
    print(f'Accuracy media {np.mean(acc):.0f}% | per-param: ' + ', '.join(f'{c}={a:.0f}%' for c,a in zip(CH,acc)))
else:
    print('S3 log assente -> scorecard accuracy saltato'); nrmse = None

In [ ]:
# Cell 4 -- SPIKE RASTER + spike-rate(t) sincronizzato allo scenario (cut-in)
import numpy as np, matplotlib.pyplot as plt
if model is not None:
    from utils.closed_loop_eval import build_scenarios
    from utils.snn_showcase import capture_run
    pgt = np.array([33.3, 1.2, 2.5, 1.1, 1.5], dtype=np.float32)
    scen = {s[0]: s for s in build_scenarios(pgt, N=400, rng=np.random.default_rng(1))}
    _, vl, s_i, v_i, cut = scen['cut_in']
    traj, spikes = capture_run(model, pgt, vl, s_i, v_i, cut_in=cut)   # spikes (T,H)
    T = spikes.shape[0]; t = np.arange(T) * 0.1
    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1, 1]})
    ys, xs = np.where(spikes.T > 0)                                    # raster
    axes[0].scatter(xs * 0.1, ys, s=6, c='k', marker='|')
    axes[0].set_ylabel('neurone hidden'); axes[0].set_title('Raster spike (cut-in): la rete "spara" piu fitto nei transitori')
    axes[1].plot(t, spikes.sum(1), color='tab:purple'); axes[1].set_ylabel('spike totali / step')
    axes[1].grid(alpha=0.3)
    axes[2].plot(t, traj['s'], label='gap [m]'); axes[2].plot(t, traj['a_ego'], label='a_ego [m/s2]')
    if cut: axes[2].axvline(cut[0]*0.1, color='r', ls='--', alpha=0.6, label='cut-in')
    axes[2].set_xlabel('t [s]'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
    axes[2].set_title('Scenario (per confronto con l attivita spiking sopra)')
    plt.tight_layout(); plt.savefig(f'{SHOW_DIR}/showcase_raster.png', dpi=120); plt.show()
    print(f'Raster: {int(spikes.sum())} spike, {(spikes.sum(0)>0).sum()}/{spikes.shape[1]} neuroni attivi')
else:
    print('[skip] modello assente'); spikes = None

In [ ]:
# Cell 5 -- ENERGIA SNN vs ANN equivalente + sparsita
import numpy as np, matplotlib.pyplot as plt
if model is not None and spikes is not None:
    from utils.snn_showcase import energy_estimate, E_MAC_FP32, E_AC_FP32
    en = energy_estimate(spikes, model)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.bar(['ANN-equiv\n(dense MAC)', 'SNN\n(event-driven)'], [en['E_ann_nJ'], en['E_snn_nJ']],
           color=['tab:gray', 'tab:green'])
    a1.set_ylabel('energia / inferenza [nJ]')
    a1.set_title(f"Energia: SNN {en['E_snn_nJ']:.0f} nJ vs ANN {en['E_ann_nJ']:.0f} nJ = {en['energy_advantage_x']:.1f}x")
    for i, val in enumerate([en['E_ann_nJ'], en['E_snn_nJ']]):
        a1.text(i, val, f'{val:.0f} nJ', ha='center', va='bottom')
    a1.grid(alpha=0.3, axis='y')
    a2.bar(['spike rate\n[%]', 'neuroni\nattivi [%]', 'vantaggio\nenergia [x]'],
           [en['mean_spike_rate_pct'], en['active_neuron_frac']*100, en['energy_advantage_x']],
           color=['tab:purple', 'tab:blue', 'tab:green'])
    a2.set_title('Metriche neuromorfiche'); a2.grid(alpha=0.3, axis='y')
    for i, val in enumerate([en['mean_spike_rate_pct'], en['active_neuron_frac']*100, en['energy_advantage_x']]):
        a2.text(i, val, f'{val:.1f}', ha='center', va='bottom')
    plt.tight_layout(); plt.savefig(f'{SHOW_DIR}/showcase_energy.png', dpi=120); plt.show()
    import json
    json.dump({k: (int(v) if isinstance(v, (int, np.integer)) else float(v)) for k, v in en.items()},
              open(f'{SHOW_DIR}/showcase_energy.json', 'w'), indent=2)
    print(f"SNN {en['E_snn_nJ']:.1f} nJ vs ANN {en['E_ann_nJ']:.1f} nJ -> {en['energy_advantage_x']:.1f}x | "
          f"sparsita {en['mean_spike_rate_pct']:.1f}% | SynOps {en['snn_synops']} vs MAC {en['ann_macs']}")
    print('NB: stima Horowitz 45nm (E_MAC=4.6pJ, E_AC=0.9pJ), per-inferenza. Po2/FPGA -> AC ancora piu economico.')
else:
    print('[skip] modello/spike assenti')

In [ ]:
# Cell 6 -- ANIMAZIONE auto (GIF) durante il cut-in
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
if model is not None:
    from utils.closed_loop_eval import build_scenarios
    from utils.snn_showcase import capture_run
    pgt = np.array([33.3, 1.2, 2.5, 1.1, 1.5], dtype=np.float32)
    scen = {s[0]: s for s in build_scenarios(pgt, N=400, rng=np.random.default_rng(1))}
    _, vl, s_i, v_i, cut = scen['cut_in']
    traj, _ = capture_run(model, pgt, vl, s_i, v_i, cut_in=cut)
    x_ego = np.cumsum(traj['v']) * 0.1
    x_lead = x_ego + traj['s']                       # leader avanti di gap
    step = max(1, len(x_ego) // 120)                 # ~120 frame
    fig, ax = plt.subplots(figsize=(11, 2.6))
    ax.set_ylim(-1, 1); ax.set_yticks([]); ax.set_xlabel('posizione [m]')
    (ego,) = ax.plot([], [], 's', ms=16, color='tab:blue', label='ego (SNN)')
    (lead,) = ax.plot([], [], 's', ms=16, color='tab:red', label='leader')
    gap_txt = ax.text(0.02, 0.85, '', transform=ax.transAxes)
    ax.legend(loc='upper right')
    def _init():
        ax.set_xlim(x_ego[0] - 10, x_lead[0] + 20); return ego, lead, gap_txt
    def _frame(k):
        i = k * step
        ego.set_data([x_ego[i]], [0]); lead.set_data([x_lead[i]], [0])
        ax.set_xlim(x_ego[i] - 10, x_lead[i] + 20)
        gap_txt.set_text(f't={i*0.1:.1f}s  gap={traj["s"][i]:.1f}m  v_ego={traj["v"][i]:.1f}')
        return ego, lead, gap_txt
    anim = animation.FuncAnimation(fig, _frame, init_func=_init, frames=len(x_ego)//step, blit=True)
    gif = f'{SHOW_DIR}/showcase_cars.gif'
    anim.save(gif, writer=animation.PillowWriter(fps=15)); plt.close(fig)
    print(f'animazione salvata: {gif}')
    from IPython.display import Image, display; display(Image(gif))
else:
    print('[skip] modello assente')

In [ ]:
# Cell 7 -- DASHBOARD riassuntivo (un colpo d'occhio)
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
lines = ['CF_FSNN — SNN car-following (PYNQ-Z1 target)', '']
lines.append(f"Parametri rete: {sum(p.numel() for p in model.parameters()) if model else 864}")
if nrmse is not None:
    lines.append(f"Accuracy identificazione (media): {np.mean([max(0,1-n)*100 for n in nrmse]):.0f}%  (5 param IDM)")
ej = f'{SHOW_DIR}/showcase_energy.json'
if os.path.isfile(ej):
    en = json.load(open(ej))
    lines.append(f"Energia/inferenza: SNN {en['E_snn_nJ']:.0f} nJ  vs  ANN {en['E_ann_nJ']:.0f} nJ  ->  {en['energy_advantage_x']:.0f}x")
    lines.append(f"Sparsita' spiking: {en['mean_spike_rate_pct']:.1f}%   (SynOps {en['snn_synops']} vs MAC {en['ann_macs']})")
safety = f'{EVAL_DIR}/safety_summary.csv'
if os.path.isfile(safety):
    sdf = pd.read_csv(safety, index_col=0)
    snn_rows = [r for r in sdf.index if r != 'oracle']
    if snn_rows:
        cr = sdf.loc[snn_rows, 'collision_rate'].max()
        lines.append(f"Sicurezza closed-loop: collision rate {cr*100:.0f}%  (0% = nessun incidente)")
else:
    lines.append("Sicurezza closed-loop: (esegui Loss_Study_Eval_ClosedLoop per il verdetto)")
fig, ax = plt.subplots(figsize=(11, 5)); ax.axis('off')
ax.text(0.5, 0.97, 'DASHBOARD CF_FSNN', ha='center', va='top', fontsize=16, weight='bold')
ax.text(0.05, 0.85, '\n'.join(lines[2:]), va='top', fontsize=12, family='monospace')
plt.savefig(f'{SHOW_DIR}/showcase_dashboard.png', dpi=120, bbox_inches='tight'); plt.show()
print('\n'.join(lines))

In [ ]:
# Cell -- push finale (eval + showcase insieme)
import subprocess
subprocess.run(['git', 'add', EVAL_DIR, SHOW_DIR], capture_output=True)
r = subprocess.run(['git', 'commit', '-m', 'Validation full: safety eval + showcase'],
                   capture_output=True, text=True)
print(r.stdout[-200:] if r.returncode == 0 else r.stderr[-200:])
subprocess.run(['git', 'push', 'origin', BRANCH], capture_output=True)
print('Validation full pushed.')